# Week 6, Lab 2 — Build / extend a local MCP server


In [1]:
WEEK = 'Week 6'
LAB = 'Lab 2 — build server'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 6 / Lab 2 — build server
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
print((ROOT / "6_mcp" / "servers" / "local_tools_server.py").read_text()[:2000])


"""Week-1 tools exposed over MCP (stdio)."""

from __future__ import annotations

import ast
import operator as op

from mcp.server.mcpserver import MCPServer

mcp = MCPServer("local-tools")

_ALLOWED = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.Mod: op.mod,
}

KB = {
    "langgraph": "LangGraph builds stateful LLM workflows as graphs of nodes and edges.",
    "ollama": "Ollama runs open-weight LLMs locally with an OpenAI-compatible API.",
    "mcp": "MCP standardizes how agents discover and call external tools and data.",
    "crewai": "CrewAI organizes agents into role-based crews.",
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED:
        return _ALLOWED[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(nod

In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import textwrap

src = textwrap.dedent(
    '''
from mcp.server.mcpserver import MCPServer
mcp = MCPServer("demo")

@mcp.tool()
def shout(text: str) -> str:
    """Uppercase the text."""
    return text.upper()

if __name__ == "__main__":
    mcp.run()
'''
)
path = ROOT / "6_mcp" / "servers" / "_demo_shout.py"
path.write_text(src, encoding="utf-8")

async def demo():
    params = StdioServerParameters(command="python", args=[str(path)])
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            print(await s.list_tools())
            print(await s.call_tool("shout", {"text": "mcp is a protocol"}))

await demo()

meta=None ttl_ms=0 cache_scope='private' next_cursor=None tools=[Tool(name='shout', title=None, description='Uppercase the text.', input_schema={'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'type': 'object', 'title': 'shoutArguments'}, execution=None, output_schema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'type': 'object', 'title': 'shoutOutput'}, icons=None, annotations=None, meta=None)] result_type='complete'
meta=None content=[TextContent(type='text', text='MCP IS A PROTOCOL', annotations=None, meta=None)] structured_content={'result': 'MCP IS A PROTOCOL'} is_error=False result_type='complete'


Add your own course topics to the KB in `local_tools_server.py`.
